# P1｜Joint Native Reference

**Pipeline ID：P1**  
**研究问题：** 在保留四个数据集原生 prediction unit、label space 与 head 的前提下，共享一个 AudioSet-only AST 表征并进行 source-proportional joint training，能否形成后续单轴 comparator 的多数据集 reference？  
**Role：** joint native reference / comparator anchor。  
**状态：Design / Not Ready。** 当前不是 Ready，不授权训练或读取 outer/test result。

## Verified Contract / Proposed Method / HOLD

**Verified Contract**
- 数据集任务必须由 native head 独立报告；missing label 不合成 negative。
- AST 初始化只能使用冻结 receipt 指定的 AudioSet-only checkpoint；不得使用 ICBHI task-selected checkpoint。
- 可复用入口候选：`baseline.shared_encoder_native_heads` 的 routed native-head 合同，以及 `baseline.four_dataset_frozen_encoder` 的四数据集 sample/native-task/verifier 基础设施。

**Proposed Method**
- 单一 AST encoder + shared adapter + 六个 dataset-native tasks；homogeneous batch 路由到合法 head。
- source-proportional sampling、native CE/BCE、无 shared-label objective，作为 P2/P3/P6/P9 的 reference anchor。

**HOLD**
- full encoder/adapter trainable scope、预算、seed、selection 与 go/no-go 尚未冻结。
- HF generic temporal `[B,L,4]`、KAUH shared mapping/diagnosis、pooled cross-dataset Score 全部 HOLD。

| 组件 | P1 recipe |
|---|---|
| Input | 四数据集合法 native unit |
| Encoder | AudioSet-only AST，具体 revision/SHA 待冻结 |
| Adapter | shared dense adapter，维度与 trainable scope 待冻结 |
| Heads | dataset-native independent heads |
| Loss | native CE/BCE；unknown/not_annotated omitted |
| Sampler | source-proportional |
| Verifier | 独立重算 per-task metrics、group overlap、label-free outer prediction lineage |

**Comparator / 唯一变化：** P1 是 anchor，不引入 eligibility objective、dataset balancing、PAFA projector 或 target-specific LODO 操作。

## 四数据集 native units / heads / split / missing-gap

| Dataset | Prediction unit | Native head | Split / grouping | Missing-gap |
|---|---|---|---|---|
| ICBHI | annotated cycle | flat4 `[B,4]` | official recording split；内部 validation patient-grouped；披露 156/218 overlap | 无标签不造 negative |
| SPRSound | annotated event | binary `[B,2]` + raw7 `[B,7]` | BioCAS2022 inter 主；intra diagnostic；patient-grouped validation | outer/inter 先 label-free prediction 后 terminal join |
| HF_Lung_V1 | 15-s recording | observed-positive phase/adventitious native presence heads | canonical date proxy grouping，明确不是 patient ID | unknown、empty annotation、time gap 不等于 negative/normal |
| KAUH/Fraiwan | recording | raw9 `[B,9]` | P-number patient-grouped；B/D/E replicas 同组 | shared mapping 与 diagnosis HOLD |


In [ ]:
import os
from pathlib import Path

PIPELINE_ID = "P1"
NOTEBOOK = Path("reproduce/P1_joint_native_reference.ipynb")
STATUS = "Design / Not Ready"
PROJECT_ROOT = Path.cwd() if Path.cwd().name != "reproduce" else Path.cwd().parent
DATASET_ROOT = Path(os.environ.get("ACOUSTIC_DATA_ROOT", "dataset/raw"))
CONFIG = Path("experiments/P1_joint_native_reference.yaml")
APPROVAL_RECEIPT = Path("result/approvals/P1_execution_authorization.json")
assert NOTEBOOK.name.startswith(f"{PIPELINE_ID}_")
assert STATUS == "Design / Not Ready"


## Checkpoint provenance / trainable scope / budget / seed / selection / execution gates

- checkpoint：必须冻结 source、revision、serialized SHA256、AudioSet-only 证明与 conversion receipt；当前 **TBD**。
- trainable scope：encoder/adapter/head 的精确集合 **TBD**；不得由 Notebook 默认。
- budget、batch policy、seed 集、validation selection、early stopping、go/no-go：均 **TBD**。
- Gate 1：P1 config 与管理批准 receipt 存在且 hash 匹配。
- Gate 2：四个 prerequisite receipts 的 unit/split/group/label/missing semantics 全部通过。
- Gate 3：本地 smoke 与独立 verifier 已冻结；outer/test 在全部条件完成前不可读。
- Gate 4：服务器执行必须另行授权。本 Notebook 始终 fail closed。

In [ ]:
required = {"config": PROJECT_ROOT / CONFIG, "approval": PROJECT_ROOT / APPROVAL_RECEIPT}
missing = [name for name, path in required.items() if not path.is_file()]
EXECUTION_ALLOWED = False
dry_run_plan = {
    "pipeline_id": PIPELINE_ID,
    "backend_candidates": ["baseline.shared_encoder_native_heads", "baseline.four_dataset_frozen_encoder"],
    "phase": "dry-run-only",
    "missing_gates": missing,
}
assert not EXECUTION_ALLOWED, "P1 skeleton must not execute training"
dry_run_plan


## Outputs / receipt schema / claim boundary

预期输出（未来获准后）：`run_manifest.json`、`protocol_and_data_receipt.json`、每 task label-free predictions、`metrics.json`、per-class/confusion tables、checkpoint lineage、`verification.json`、`decision_receipt.json`。所有 receipt 必须含 Pipeline ID、config/protocol hash、dataset/version/unit、split/grouping、checkpoint SHA、trainable scope、budget、seed、selection、outer-label join 时点与 warning count。

**Claim boundary：** 仅可形成 target-supervised multi-dataset joint native reference；不是 zero-shot、不是 clean generalization、不是 universal shared-label success，也不代表 full-encoder evidence。

**Test Result=Not run**  
**Decision：Not evaluated；Design / Not Ready。**